# Can AI select fillers based on verbal description?
Chicago Face Database + CLIP Pilot Study

- The pilot used `CFD Version 3.0` neutral-expression images as the retrieval gallery.
- Images were indexed using the sentence-transformers CLIP model `clip-ViT-B-32`.
- Both image and text embeddings were `L2-normalized`, and retrieval used `cosine similarity` via inner product.
- Proxy descriptions were generated from CFD age, race, and gender metadata using a fixed natural-language template.

The pilot study is designed to answer two feasibility questions:

1. Can CLIP retrieve the corresponding CFD target from a text description?
2. Can CLIP generate stable top-5 candidate fillers for each target/description?

Environment can be created with the code here in PowerShell:

```powershell
python -m pip install -e ".[cfd-pilot]"
python -c "import sentence_transformers, torch, openpyxl; print('CLIP pilot deps OK')"
```

## 0. Paths And Dependency Check

Prepares the output directory, and checks the dependencies required for the study.

In [ ]:
from pathlib import Path
import importlib.util
import json
import sys

import pandas as pd

# The current working directory may be either the repository root or the studies folder.
cwd = Path.cwd().resolve()
ROOT = cwd if (cwd / "src" / "pyWitnessAI").exists() else cwd.parent
SRC = ROOT / "src"
OUT = ROOT / "build" / "study3_cfd_clip_pilot_outputs"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

OUT.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("OUT:", OUT)

for package in ["sentence_transformers", "torch", "openpyxl", "PIL", "pandas"]:
    print(f"{package:22s}", "OK" if importlib.util.find_spec(package) else "MISSING")

In [ ]:
import pandas as pd

from pyWitnessAI.cfd_clip_pilot import (
    ClipIndex,
    build_cfd_manifest,
    build_filler_sets,
    build_proxy_descriptions,
    evaluate_retrieval,
)
from pyWitnessAI.cfd_clip_pilot.clip_backend import SentenceTransformerClipEncoder

CFD_IMAGE_DIR = ROOT / "data" / "CFD Version 3.0" / "Images"
CFD_METADATA_PATH = ROOT / "data" / "CFD Version 3.0" / "CFD 3.0 Norming Data and Codebook.xlsx"

MODEL_NAME = "clip-ViT-B-32"
DEVICE = "cpu"  # Change to "cuda" if a CUDA-enabled PyTorch installation is available.
SEED = 2026

print("CFD_IMAGE_DIR exists:", CFD_IMAGE_DIR.exists())
print("CFD_METADATA_PATH exists:", CFD_METADATA_PATH.exists())

## 1. Build The Manifest

`manifest.csv` is the master index for the CFD gallery. Each row corresponds to one neutral face image and is merged with metadata from the official CFD norming workbook.

Core columns include:

- `image_id`: full image identifier, for example `CFD-AF-200-228-N`
- `target_id`: model identifier aligned with the CFD workbook, for example `AF-200`
- `expression`: expression code; this pilot keeps neutral images (`N`) by default
- `image_path`: image file path
- `gender`, `race`, `age`: standardized helper fields derived from the CFD metadata

The manifest also preserves the original CFD ratings and facial measurements for later diagnostic or fairness-oriented analyses.

In [ ]:
manifest = build_cfd_manifest(
    image_dir=CFD_IMAGE_DIR,
    metadata_path=CFD_METADATA_PATH,
    neutral_only=True,
)

manifest_path = OUT / "manifest.csv"
manifest.to_csv(manifest_path, index=False)

print("Saved:", manifest_path)
print("shape:", manifest.shape)
manifest[["image_id", "target_id", "expression", "gender", "race", "age", "image_path"]].head()

In [ ]:
summary_manifest = {
    "rows": len(manifest),
    "unique_targets": int(manifest["target_id"].nunique()),
    "missing_gender": int(manifest["gender"].isna().sum()),
    "missing_race": int(manifest["race"].isna().sum()),
    "missing_age": int(manifest["age"].isna().sum()),
}
summary_manifest

In [ ]:
manifest["race"].fillna("<missing>").value_counts().rename("n")

## 2. Generate Proxy Descriptions

`queries_complete.csv` is not a set of real eyewitness descriptions. It is a proxy query table generated from CFD metadata.

For example, the row for `AF-200` uses:

- `age = 32.57...`
- `race = Asian`
- `gender = female`

and renders the following CLIP-friendly sentence:

`A neutral frontal face photograph of around 32.6 years old asian female person.`

`required_fields=["age", "gender", "race"]` excludes targets with incomplete metadata.

In [ ]:
queries = build_proxy_descriptions(
    manifest,
    required_fields=["age", "gender", "race"],
)

queries_path = OUT / "queries_complete.csv"
queries.to_csv(queries_path, index=False)

print("Saved:", queries_path)
print("shape:", queries.shape)
queries[["query_id", "target_id", "image_id", "description"]].head(10)

## 3. Build The Full CLIP Index

If the full index already exists from a PowerShell run, this cell loads it. To force a rebuild, set `REBUILD_FULL_INDEX = True`.

In [ ]:
REBUILD_FULL_INDEX = False
full_index_dir = OUT / "clip_index"

if (full_index_dir / "image_embeddings.npy").exists() and not REBUILD_FULL_INDEX:
    clip_index = ClipIndex.load(full_index_dir)
    print("Loaded existing full index:", full_index_dir)
else:
    clip_index = ClipIndex.build(
        manifest=manifest,
        encoder=encoder,
        batch_size=32,
        show_progress=True,
    )
    clip_index.save(full_index_dir)
    print("Saved full index:", full_index_dir)

clip_index.metadata

## 4. Full Retrieval Evaluation

This step retrieves the top-k CFD images for each proxy description and checks whether the true `target_id` appears among the returned results.

In [ ]:
encoder = SentenceTransformerClipEncoder(model_name=MODEL_NAME, device=DEVICE)

retrieval_results, per_query, summary = evaluate_retrieval(
    index=clip_index,
    queries=queries,
    encoder=encoder,
    top_k=50,
)

eval_dir = OUT / "evaluation"
eval_dir.mkdir(parents=True, exist_ok=True)
retrieval_results.to_csv(eval_dir / "retrieval_results.csv", index=False)
per_query.to_csv(eval_dir / "per_query_metrics.csv", index=False)
(eval_dir / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

summary

In [ ]:
pd.DataFrame(
    {
        "metric": ["top_1", "top_5", "top_10", "top_50"],
        "hit_rate": [
            per_query["hit_top_1"].mean(),
            per_query["hit_top_5"].mean(),
            per_query["hit_top_10"].mean(),
            per_query["hit_top_50"].mean(),
        ],
        "n_hit": [
            int(per_query["hit_top_1"].sum()),
            int(per_query["hit_top_5"].sum()),
            int(per_query["hit_top_10"].sum()),
            int(per_query["hit_top_50"].sum()),
        ],
    }
)

In [ ]:
per_query[["query_id", "target_id", "description", "target_rank", "hit_top_50"]].head(10)

## 5. Metadata-Stratified Diagnostics

These summaries are pilot diagnostics, not formal fairness conclusions. The descriptions are metadata-generated proxy text rather than real eyewitness descriptions.

In [ ]:
per_query.groupby("gender").agg(
    n=("query_id", "count"),
    hit_top_5=("hit_top_5", "mean"),
    hit_top_10=("hit_top_10", "mean"),
    hit_top_50=("hit_top_50", "mean"),
    median_rank=("target_rank", "median"),
).reset_index()

In [ ]:
per_query.groupby("race").agg(
    n=("query_id", "count"),
    hit_top_5=("hit_top_5", "mean"),
    hit_top_10=("hit_top_10", "mean"),
    hit_top_50=("hit_top_50", "mean"),
    median_rank=("target_rank", "median"),
).sort_values("hit_top_50", ascending=False).reset_index()

## 6. Export Top-5 Fillers

Excludes the same `target_id` by default so that the target is not selected as a filler.

In [ ]:
fillers = build_filler_sets(
    index=clip_index,
    queries=queries,
    encoder=encoder,
    top_k=50,
    filler_count=5,
)

fillers_path = OUT / "filler_sets.csv"
fillers.to_csv(fillers_path, index=False)

print("Saved:", fillers_path)
print("shape:", fillers.shape)
fillers.head(5)

In [ ]:
# Check whether every query received exactly five fillers.
fillers.groupby("query_id").size().value_counts().rename("n_queries")

## 7. Inspect One Target

Example: `AF-200`.

In [ ]:
TARGET_ID = "AF-200"

print("Query row")
display(queries.loc[queries["query_id"] == TARGET_ID, ["query_id", "target_id", "image_id", "description"]])

print("Evaluation row")
display(per_query.loc[per_query["query_id"] == TARGET_ID, ["query_id", "target_id", "target_rank", "hit_top_50"]])

print("Top retrievals")
display(retrieval_results.loc[retrieval_results["query_query_id"] == TARGET_ID].head(5))

print("Selected fillers")
display(fillers.loc[fillers["query_id"] == TARGET_ID])

## 8. Print The Top-k Images of Certain Description

In [ ]:
from pyWitnessAI.cfd_clip_pilot import ClipIndex
from pyWitnessAI.cfd_clip_pilot.clip_backend import SentenceTransformerClipEncoder

index = ClipIndex.load(
    r"D:\PhD\Software\pyWitnessAI\build\study3_cfd_clip_pilot_outputs\clip_index"
)

encoder = SentenceTransformerClipEncoder(
    model_name="clip-ViT-B-32",
    device="cpu",
)

description = "A young African woman with long hair."

results = index.search_texts(
    [description],
    encoder=encoder,
    top_k=10,
)

results

In [ ]:
from IPython.display import display
from PIL import Image, ImageOps
import numpy as np

def resolve_repo_path(path_like, root=ROOT):
    path = Path(path_like)
    if path.is_absolute():
        return path
    return root / path
    
for _, row in results.iterrows():
    print(
        f"Rank {row['rank']} | "
        f"target_id={row['target_id']} | "
        f"score={row['clip_score']:.4f}"
    )
    image_path = resolve_repo_path(row['image_path'])
    img = Image.open(image_path)
    img = ImageOps.exif_transpose(img).convert("RGB")
    img_clean = Image.fromarray(np.array(img, dtype=np.uint8), mode="RGB")
    display(img_clean.resize((180, 120)))
    # display(Image.open(image_path))

In [ ]:
from pyWitnessAI.cfd_clip_pilot.visualization import search_and_show_lineup
from pyWitnessAI.cfd_clip_pilot import (
    ClipIndex,
    build_cfd_manifest,
    build_filler_sets,
    build_proxy_descriptions,
    evaluate_retrieval,
)

description = "A neutral frontal face photograph of around 32 years old asian female person."

manifest = build_cfd_manifest(
    image_dir=CFD_IMAGE_DIR,
    metadata_path=CFD_METADATA_PATH,
    neutral_only=True,
)

index = ClipIndex.load(
    r"D:\PhD\Software\pyWitnessAI\build\study3_cfd_clip_pilot_outputs\clip_index"
)

queries = build_proxy_descriptions(
    manifest,
    required_fields=["age", "gender", "race"],
)

target_row = queries.loc[queries["query_id"] == "AF-200"].iloc[0]

results, fig = search_and_show_lineup(
    index=index,
    encoder=encoder,
    description=description,
    target_image_path=target_row["image_path"],
    root=ROOT,
    top_k=9,
    cols=3,
    target_position="left",
)

In [ ]:
from pyWitnessAI.cfd_clip_pilot.visualization import show_clip_lineup

one_query_fillers = fillers.loc[fillers["query_id"] == "BF-001"]

target_row = queries.loc[queries["query_id"] == "BF-001"].iloc[0]

fig = show_clip_lineup(
    one_query_fillers,
    target_image_path=target_row["image_path"],
    root=ROOT,
    top_k=5,
    cols=3,
    image_path_col="filler_image_path",
    rank_col="filler_position",
    score_col="clip_score",
    id_col="filler_target_id",
    target_position="left",
    suptitle="AF-200 candidate fillers",
)

## 10. Next step: Different Models, Real Human Descriptions, Add ArcFace

We can do comparisons:

- clip-ViT-B-32 vs ViT-L/14
- top_k = 20, 50, 100
- raw metadata proxy descriptions vs real human descriptions
- with vs without diversity filtering
- CLIP-only vs CLIP + ArcFace reranking

For a real-description pilot, prepare `queries_real.csv` with at least these columns:

- `query_id`: unique ID for each description
- `participant_id`: participant ID
- `target_id`: corresponding CFD target, for example `AF-200`
- `image_id`: corresponding CFD image ID; optional but recommended
- `description`: the participant's free-recall description

If the target is not in the CFD gallery, target-rank evaluation is not possible; in that case, export fillers and evaluate filler-description match with human ratings.

In [ ]:
# real_template_path = OUT / "queries_real_template.csv"

# template = pd.DataFrame(
#     columns=["query_id", "participant_id", "target_id", "image_id", "description"]
# )
# template.to_csv(real_template_path, index=False)

# print("Saved real-description template:", real_template_path)
# template

Save real participant descriptions as:

`build/study3_cfd_clip_pilot_outputs/queries_real.csv`

Then uncomment and run the code below.

In [ ]:
# real_queries_path = OUT / "queries_real.csv"
# real_queries = pd.read_csv(real_queries_path)
#
# retrieval_real, per_query_real, summary_real = evaluate_retrieval(
#     index=clip_index,
#     queries=real_queries,
#     encoder=encoder,
#     top_k=50,
# )
#
# eval_real_dir = OUT / "evaluation_real"
# eval_real_dir.mkdir(parents=True, exist_ok=True)
# retrieval_real.to_csv(eval_real_dir / "retrieval_results.csv", index=False)
# per_query_real.to_csv(eval_real_dir / "per_query_metrics.csv", index=False)
# (eval_real_dir / "summary.json").write_text(json.dumps(summary_real, indent=2), encoding="utf-8")
#
# summary_real

In [ ]:
# fillers_real = build_filler_sets(
#     index=clip_index,
#     queries=real_queries,
#     encoder=encoder,
#     top_k=50,
#     filler_count=5,
# )
# fillers_real.to_csv(OUT / "filler_sets_real.csv", index=False)
# fillers_real.head(10)

## 11. CLIP Feature Ladder

Define a target face, define a cumulative description ladder, and display CLIP top-k lineups as the description becomes more specific.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from pyWitnessAI.cfd_clip_pilot import (
    ClipRetrievalProbe,
    DescriptionLadder,
    build_cfd_manifest,
)
from pyWitnessAI.cfd_clip_pilot.clip_backend import SentenceTransformerClipEncoder

VISUAL_TARGET_ID = "AM-247"
VISUAL_TOP_K = 9
VISUAL_COLS = 3
VISUAL_BATCH_SIZE = BATCH_SIZE if "BATCH_SIZE" in globals() else 32

if "manifest" not in globals():
    manifest = build_cfd_manifest(
        image_dir=CFD_IMAGE_DIR,
        metadata_path=CFD_METADATA_PATH,
        neutral_only=True,
    )

if "encoder" not in globals():
    encoder = SentenceTransformerClipEncoder(model_name=MODEL_NAME, device=DEVICE)

visual_probe = ClipRetrievalProbe.from_manifest(
    manifest=manifest.drop_duplicates("target_id", keep="first").reset_index(drop=True),
    index_dir=OUT / "clip_index",
    encoder=encoder,
    root=ROOT,
    batch_size=VISUAL_BATCH_SIZE,
)

visual_target_row = visual_probe.find_target(target_id=VISUAL_TARGET_ID)
visual_target_row[["target_id", "image_id", "gender", "race", "age", "image_path"]]


In [ ]:
visual_ladder = DescriptionLadder.from_steps(
    [
        ("gender", "male", "subject"),
        ("hair", "puffy dark hair with a central parting", "detail"),
        ("facial_hair", "a full beard neatly shaven", "detail"),
        ("eyes", "dark eyes", "detail"),
        ("eyebrows", "thick eyebrows", "detail"),
        ("nose", "a broad nose", "detail"),
        ("build", "a broad build", "detail"),
        ("face_shape", "a round face", "detail"),
        ("race", "Asian", "subject"),
        ("forehead", "a high forehead", "detail"),
        ("mouth", "full lips", "detail"),
        ("ears", "visible ears", "detail"),
        ("jaw", "a strong jaw", "detail"),
        ("teeth", "hiden teeth", "detail"),
    ],
    prefix="A neutral frontal face photograph of",
)

visual_ladder_frame = visual_ladder.to_frame()
visual_ladder_frame


In [ ]:
VISUAL_STAGE_TO_SHOW = "teeth"

visual_stage_results, visual_stage_fig = visual_probe.show_stage(
    visual_ladder,
    VISUAL_STAGE_TO_SHOW,
    target_id=VISUAL_TARGET_ID,
    top_k=VISUAL_TOP_K,
    cols=VISUAL_COLS,
    display_heading=True,
    display_figure=True,
    show_description_title=False,
    show_table=True,
)
plt.close(visual_stage_fig)


In [ ]:
RUN_ALL_VISUAL_STEPS = True

visual_all_results = []
if RUN_ALL_VISUAL_STEPS:
    for feature in visual_ladder_frame["feature"]:
        stage_results, stage_fig = visual_probe.show_stage(
            visual_ladder,
            feature,
            target_id=VISUAL_TARGET_ID,
            top_k=VISUAL_TOP_K,
            cols=VISUAL_COLS,
            display_heading=True,
            display_figure=True,
            show_description_title=False,
            show_table=False,
        )
        plt.close(stage_fig)
        visual_all_results.append(stage_results)

visual_all_results = pd.concat(visual_all_results, ignore_index=True) if visual_all_results else pd.DataFrame()
visual_lineup_results_path = OUT / "visual_feature_ladder_lineup_results.csv"
visual_all_results.to_csv(visual_lineup_results_path, index=False)
print("Saved:", visual_lineup_results_path)
visual_all_results.head()


## 12. Visual CLIP Feature Ladder With Expression Images

This section repeats the visual probe on the expressive CFD gallery. It builds or loads a separate CLIP index for non-neutral expression images, then uses `WF-025-006-HO` as the target image.


In [ ]:
from pyWitnessAI.cfd_clip_pilot import filter_expression_images

EXPRESSION_TARGET_IMAGE_ID = "WF-025-006-HO"
EXPRESSION_TOP_K = 9
EXPRESSION_COLS = 3
EXPRESSION_INDEX_DIR = OUT / "clip_index_expressions_only"
EXPRESSION_MANIFEST_PATH = OUT / "manifest_expressions_only.csv"

all_expression_manifest = build_cfd_manifest(
    image_dir=CFD_IMAGE_DIR,
    metadata_path=CFD_METADATA_PATH,
    neutral_only=False,
)
expression_manifest = filter_expression_images(all_expression_manifest)
expression_manifest.to_csv(EXPRESSION_MANIFEST_PATH, index=False)

expression_probe = ClipRetrievalProbe.from_manifest(
    manifest=expression_manifest,
    index_dir=EXPRESSION_INDEX_DIR,
    encoder=encoder,
    root=ROOT,
    batch_size=VISUAL_BATCH_SIZE,
    show_progress=True,
)

expression_target_row = expression_probe.find_target(image_id=EXPRESSION_TARGET_IMAGE_ID)
print("all CFD images:", len(all_expression_manifest))
print("non-neutral expression images:", len(expression_manifest))
print("saved:", EXPRESSION_MANIFEST_PATH)
expression_target_row[["image_id", "target_id", "expression", "gender", "race", "age", "image_path"]]


In [ ]:
expression_ladder = DescriptionLadder.from_steps(
    [
        ("gender", "female", "subject"),
        ("hair", "tied-back long brown hair", "detail"),
        ("facial_hair", "no facial hair", "detail"),
        ("eyes", "blue eyes", "detail"),
        ("eyebrows", "thin arched eyebrows", "detail"),
        ("nose", "a narrow nose", "detail"),
        ("build", "a slim build", "detail"),
        ("face_shape", "an oval face", "detail"),
        ("race", "White", "subject"),
        ("forehead", "a high forehead", "detail"),
        ("mouth", "a wide smiling mouth", "detail"),
        ("ears", "partly visible ears", "detail"),
        ("jaw", "a soft jawline", "detail"),
        ("teeth", "visible white teeth", "detail"),
    ],
    base_details=("a mouth-opened happy smile",),
    base_feature="expression_only",
)

expression_ladder_frame = expression_ladder.to_frame()
expression_ladder_frame


In [ ]:
EXPRESSION_STAGE_TO_SHOW = "teeth"

expression_stage_results, expression_stage_fig = expression_probe.show_stage(
    expression_ladder,
    EXPRESSION_STAGE_TO_SHOW,
    target_image_id=EXPRESSION_TARGET_IMAGE_ID,
    top_k=EXPRESSION_TOP_K,
    cols=EXPRESSION_COLS,
    candidate_columns=("expression",),
    display_heading=True,
    display_figure=True,
    show_description_title=False,
    show_table=True,
)
plt.close(expression_stage_fig)


In [ ]:
RUN_ALL_EXPRESSION_STEPS = True

expression_all_results = []
if RUN_ALL_EXPRESSION_STEPS:
    for feature in expression_ladder_frame["feature"]:
        stage_results, stage_fig = expression_probe.show_stage(
            expression_ladder,
            feature,
            target_image_id=EXPRESSION_TARGET_IMAGE_ID,
            top_k=EXPRESSION_TOP_K,
            cols=EXPRESSION_COLS,
            candidate_columns=("expression",),
            display_heading=True,
            display_figure=True,
            show_description_title=False,
            show_table=False,
        )
        plt.close(stage_fig)
        expression_all_results.append(stage_results)

expression_all_results = pd.concat(expression_all_results, ignore_index=True) if expression_all_results else pd.DataFrame()
expression_lineup_results_path = OUT / "visual_expression_feature_ladder_lineup_results.csv"
expression_all_results.to_csv(expression_lineup_results_path, index=False)
print("Saved:", expression_lineup_results_path)
expression_all_results.head()
